In [ ]:
# --- repo bootstrap (auto-added) ---
# Run paths relative to the repo root and make src/ importable.
import os, sys
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
_SRC = os.path.abspath("src")
if _SRC not in sys.path:
    sys.path.insert(0, _SRC)


In [1]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

In [2]:
from datasets import load_dataset
import pandas as pd


## Loading the MGSD dataset.

dataset = load_dataset("wu981526092/MGSD")

data = dataset['train']
df = data.to_pandas()


## Loading the MentalManip dataset

dataset_2 = load_dataset("audreyeleven/MentalManip", "mentalmanip_maj")
data_2 = dataset_2["train"]
df_2 = data_2.to_pandas()

Some datasets params were ignored: ['license']. Make sure to use only valid params for the dataset builder and to have a up-to-date version of the `datasets` library.


In [3]:
from data_loader import load_mgsd_dataset, load_mentalmanip_dataset

sample_sizes_mgsd = {
    'stereotype': 250,
    'unrelated': 250,
}

sample_size_examples_mgsd = {
    'stereotype': 5,
    'unrelated': 5
}

sample_sizes_manip = {1: 250, 0: 250}
sample_sizes_examples_manip = {1: 5, 0: 5}
max_len_examples = 1000

sample_mgsd, sample_examples_mgsd = load_mgsd_dataset(
    df, 
    sample_sizes_mgsd, 
    sample_size_examples_mgsd,
    random_state=42,
    random_state_examples=0,
    )

sample_mentalmanip, sample_examples_mentalmanip = load_mentalmanip_dataset(
    df_2, 
    sample_sizes_manip, 
    sample_sizes_examples_manip, 
    max_len_examples,
    random_state=42,
    random_state_examples=0,
    )


print("MGSD Test set balance:\n", sample_mgsd["label"].value_counts())
print("MGSD Few-shot examples balance:\n", sample_examples_mgsd["label"].value_counts())

print("MentalManip Test set balance:\n", sample_mentalmanip["manipulative"].value_counts())
print("MentalManip Few-shot examples balance:\n", sample_examples_mentalmanip["manipulative"].value_counts())

MGSD Test set balance:
 label
unrelated     250
stereotype    250
Name: count, dtype: int64
MGSD Few-shot examples balance:
 label
stereotype    5
unrelated     5
Name: count, dtype: int64
MentalManip Test set balance:
 manipulative
1    250
0    250
Name: count, dtype: int64
MentalManip Few-shot examples balance:
 manipulative
1    5
0    5
Name: count, dtype: int64


In [4]:
from dotenv import load_dotenv

import os
import openai
from openai import OpenAI
from anthropic import Anthropic
from mistralai import Mistral
import cohere
import google.generativeai as genai
from xai_sdk import Client as XAIClient


load_dotenv()


ENV_VARS = {
    "API_KEY_OPENAI": "OpenAI",
    "API_KEY_DEEPSEEK": "DeepSeek",
    "API_KEY_GROK": "Grok",
    "API_KEY_ANTHROPIC": "Anthropic",
    "API_KEY_GEMINI": "Gemini",
    "API_KEY_MISTRAL": "Mistral",
    "API_KEY_COHERE": "Cohere",
}

for var, name in ENV_VARS.items():
    if not os.getenv(var):
        print(f"Warning: {name} - API key missing in .env file.")
        continue

API_KEY_OPENAI = os.getenv("API_KEY_OPENAI")
API_KEY_DEEPSEEK = os.getenv("API_KEY_DEEPSEEK")
API_KEY_ANTHROPIC = os.getenv("API_KEY_ANTHROPIC")
API_KEY_GEMINI = os.getenv("API_KEY_GEMINI")
API_KEY_MISTRAL = os.getenv("API_KEY_MISTRAL")
API_KEY_COHERE = os.getenv("API_KEY_COHERE")
API_KEY_GROK = os.getenv("API_KEY_GROK")


backend_to_run = [
    "openai-4.1-mini",
    #"openai-4o-mini",
    #"mistral-small-2506",
    #"mistral-small-2503",
    #"anthropic-sonnet",
    #"deepseek-v3-chat",
]

backends = {
    "openai-4.1-mini": {
        "provider": "openai",
        "client":  OpenAI(api_key=API_KEY_OPENAI),
        "model":   "gpt-4.1-mini-2025-04-14",
        "fname":   "openai_4.1_mini"
    },
    "openai-4o-mini": {
        "provider": "openai",
        "client":  OpenAI(api_key=API_KEY_OPENAI),
        "model":   "gpt-4o-mini-2024-07-18",
        "fname":   "openai_4o_mini"
    },

    "deepseek-v3-chat": {
        "provider": "openai",
        "client":  OpenAI(api_key=API_KEY_DEEPSEEK, base_url="https://api.deepseek.com"),
        "model":   "deepseek-chat",
        "fname":   "deepseek_v3"
    },

    "anthropic-sonnet": {
        "provider": "anthropic",
        "client":  Anthropic(api_key=API_KEY_ANTHROPIC),
        "model":   "claude-3-7-sonnet-latest",
        "fname":   "anthropic_3_7_sonnet"
    },

    "gemini-2.5-flash": {
        "provider": "gemini",
        "client":  (genai.configure(api_key=API_KEY_GEMINI) or genai.GenerativeModel("gemini-2.5-flash")),
        "model":   "gemini-2.5-flash",
        "fname":   "google_gemini_2_5_flash"
    },

    "mistral-small-2506": {
        "provider": "mistral",
        "client":  Mistral(api_key=API_KEY_MISTRAL),
        "model":   "mistral-small-2506",
        "fname":   "mistral_small_2506"
    },
    "mistral-small-2503": {
        "provider": "mistral",
        "client":  Mistral(api_key=API_KEY_MISTRAL),
        "model":   "mistral-small-2503",
        "fname":   "mistral_small_2503"
    },
}


## Chain of Thoughts

In [6]:
from tqdm import tqdm
import os, json, re
import pandas as pd
from chain_of_thought import ChainOfThoughts
from stereotype_definitions import stereotype_definition_short_binary
from manipulation_definitions import manipulation_definition_short
from sklearn.metrics import classification_report, confusion_matrix
from cases.stereotypes_case import stereotypes_case
from cases.manipulation_case import manipulation_case
from cases.mmlu_case import mmlu_case

case_name_set = ["stereotype"]  # "stereotype", "manipulation", "mmlu" 
strategy = "optimized"          # "optimized"
max_tokens = 700

for backend in backend_to_run:
    
    model = backends[backend]["model"]
    client = backends[backend]["client"]
    model_filename = backends[backend]["fname"]

    for case_name in case_name_set:
        print(f"\n=== Running Chain of Thought for {case_name} ({strategy}) ===")

        if case_name.lower() == "manipulation":
            case = manipulation_case
            task_definition = manipulation_definition_short

            data = sample_mentalmanip.copy()

        elif case_name.lower() == "stereotype":
            case = stereotypes_case
            task_definition = stereotype_definition_short_binary

            data = sample_mgsd.copy()

        else:
            raise ValueError(f"Unknown case name: {case_name}")

        cot_classifier = ChainOfThoughts(
            case=case,
            client=client,
            model=model,
            max_tokens=max_tokens,
            task_definition=task_definition,
        )

        rows = []
        detailed_reasoning = []

        for idx, row in tqdm(data.iterrows(), total=len(data), desc=f"Processing {case_name}"):
            text = row[case.input_col]
            true_label = row[case.label_col]
            if isinstance(true_label, str):
                true_label = true_label.strip()

            try:
                predicted_label, metrics = cot_classifier.classify_with_strategy(
                    text, strategy=strategy
                )

                mapped_label = case.label_map.get(
                    predicted_label.strip(), list(case.label_map.values())[-1]
                )

                results = {
                    "sample_id": idx,
                    "text": text,
                    "true_label": true_label,
                    "pred_label": mapped_label,
                    "raw_pred_label": predicted_label,
                    "max_tokens": cot_classifier.max_tokens,
                    "tokens_used": metrics.get("tokens_used"),
                    "prompt_tokens": metrics.get("prompt_tokens"),
                    "completion_tokens": metrics.get("completion_tokens"),
                    "latency": metrics.get("latency"),
                    "strategy": strategy,
                }
                rows.append(results)

                raw_resp = metrics.get("raw_response", "")
                steps, analysis, final_line = cot_classifier._parse_steps_and_final(raw_resp)
                reasoning_detail = {
                    "sample_id": idx,
                    "raw_response": raw_resp,
                    "parsed_steps": steps,
                    "analysis": analysis,       
                    "final_line": final_line,
                    "final_label": predicted_label,
                    "mapped_label": mapped_label,
                }
                detailed_reasoning.append(reasoning_detail)

            except Exception as e:
                print(f"Error processing sample {idx}: {e}")
                continue

        if not rows:
            print(f"No successful classifications for {case_name}")
            continue

        output_file = f"results/{model_filename}/cot/classic/results_{case_name.lower()}_{strategy}_cot.csv"
        reasoning_file = f"results/{model_filename}/cot/classic/reasoning_{case_name.lower()}_{strategy}_cot.json"

        os.makedirs(os.path.dirname(output_file), exist_ok=True)

        df_out = pd.DataFrame(rows)
        df_out.to_csv(output_file, index=False)
        print(f"=== Saved {len(df_out)} rows to {output_file} ===")

        with open(reasoning_file, 'w', encoding='utf-8') as f:
            json.dump(detailed_reasoning, f, indent=2, ensure_ascii=False)
        print(f"=== Saved detailed reasoning to {reasoning_file} ===")

        try:
            if case_name.lower() == "manipulation":
                y_true = df_out["true_label"].astype(int)
                y_pred = df_out["pred_label"].astype(int)
            elif case_name.lower() == "stereotype":
                y_true = df_out["true_label"].astype(str).str.strip().str.lower()
                y_pred = df_out["pred_label"].astype(str).str.strip().str.lower()


            print(f"\n=== Classification Report for {case_name} ({strategy}) ===")
            print(classification_report(y_true, y_pred, zero_division=0))
            print(f"\n=== Confusion Matrix for {case_name} ===")
            labels = sorted(set(y_true) | set(y_pred))
            print(pd.DataFrame(confusion_matrix(y_true, y_pred, labels=labels), index=labels, columns=labels))

            accuracy = (y_true == y_pred).mean()
            print(f"\n=== Accuracy for {case_name}: {accuracy:.2%} ===")

            print(f"\n=== Label Distribution ===")
            print("True labels:")
            print(pd.Series(y_true).value_counts())
            print("Predicted labels:")
            print(pd.Series(y_pred).value_counts())

        except Exception as e:
            print(f"Error in evaluation for {case_name}: {e}")

        metrics = cot_classifier.get_metrics()
        print(f"\n=== CoT Performance Metrics ===")
        print(f"- Avg tokens per call: {metrics['avg_tokens_per_call']:.1f}")
        print(f"- Avg latency per call: {metrics['avg_latency_per_call']:.2f}s")
        print(f"- Total calls: {metrics['total_calls']}")


        print(f"\n=== Sample Reasoning (raw/parsed) ===")
        for i, reasoning in enumerate(detailed_reasoning[:3]):
            print(f"\nSample {reasoning['sample_id']}:")
            print(f"True Label: {data.loc[reasoning['sample_id']][case.label_col]}")
            print(f"Predicted: {reasoning['final_label']} -> Mapped: {reasoning['mapped_label']}")
            if reasoning['parsed_steps']:
                print("Parsed Steps:")
                for step in reasoning['parsed_steps']:
                    print(f"  Step {step['step']}: {step['content'][:150]}...")
            else:
                print("No explicit 'Step N' blocks found.")
            if reasoning['final_line']:
                print(f"Final Line: {reasoning['final_line']}")
            print("Raw Response (truncated):")
            print((reasoning['raw_response'] or "")[:300] + "...")
            print("-" * 50)

#except Exception as e:
#    print(f"== Error: {e} ==")
#    import traceback
#    traceback.print_exc()

print("\n=== Chain of Thought Evaluation Complete ===")



=== Running Chain of Thought for stereotype (optimized) ===


Processing stereotype: 100%|██████████| 500/500 [21:56<00:00,  2.63s/it]


=== Saved 500 rows to results/openai_4.1_mini/cot/classic/results_stereotype_optimized_cot.csv ===
=== Saved detailed reasoning to results/openai_4.1_mini/cot/classic/reasoning_stereotype_optimized_cot.json ===

=== Classification Report for stereotype (optimized) ===
              precision    recall  f1-score   support

  stereotype       0.73      0.74      0.74       250
   unrelated       0.74      0.72      0.73       250

    accuracy                           0.73       500
   macro avg       0.73      0.73      0.73       500
weighted avg       0.73      0.73      0.73       500


=== Confusion Matrix for stereotype ===
            stereotype  unrelated
stereotype         186         64
unrelated           70        180

=== Accuracy for stereotype: 73.20% ===

=== Label Distribution ===
True labels:
true_label
unrelated     250
stereotype    250
Name: count, dtype: int64
Predicted labels:
pred_label
stereotype    256
unrelated     244
Name: count, dtype: int64

=== CoT Perfor

## Role playing with Chain-of-Thoughts

In [ ]:
from tqdm import tqdm
import os, json, re
import pandas as pd
from chain_of_thought import ChainOfThoughts
from stereotype_definitions import stereotype_definition_short_binary
from manipulation_definitions import manipulation_definition_short
from sklearn.metrics import classification_report, confusion_matrix
from cases.stereotypes_case import stereotypes_case
from cases.manipulation_case import manipulation_case
from profiles.profile_sets import PERSON_ETHNICS

case_name_set = ["stereotype"]  # "stereotype", "manipulation"
strategy = "optimized"         
max_tokens = 700
role_playing_mode = "passive"
selected_profiles = ["profile1", "profile2", "profile3"] 

os.makedirs(f"results/{model_filename}/cot/reasoning", exist_ok=True)

try:
    for profile_n in selected_profiles:
        for case_name in case_name_set:
            print(f"\n=== Running Chain of Thought for {case_name} ({strategy}) - {profile_n} ===")
    
            if case_name.lower() == "manipulation":
                case = manipulation_case
                task_definition = manipulation_definition_short
                data = sample_mentalmanip
            elif case_name.lower() == "stereotype":
                case = stereotypes_case
                task_definition = stereotype_definition_short_binary
                data = sample_mgsd
            else:
                raise ValueError(f"Unknown case name: {case_name}")
    
            cot_classifier = ChainOfThoughts(
                case=case,
                client=client,
                model=model,
                max_tokens=max_tokens,
                task_definition=task_definition,
                person_key=profile_n,
                role_playing=role_playing_mode, 
                person_set=PERSON_ETHNICS
            )
    
            rows = []
            detailed_reasoning = []
    
            for idx, row in tqdm(data.iterrows(), total=len(data), desc=f"Processing {case_name}"):
                text = row[case.input_col]
                true_label = row[case.label_col]
                if isinstance(true_label, str):
                    true_label = true_label.strip()
    
                try:
                    predicted_label, metrics = cot_classifier.classify_with_strategy(
                        text, strategy=strategy
                    )
    
                    mapped_label = case.label_map.get(
                        predicted_label.strip(), list(case.label_map.values())[-1]
                    )
    
                    results = {
                        "sample_id": idx,
                        "text": text,
                        "true_label": true_label,
                        "pred_label": mapped_label,
                        "raw_pred_label": predicted_label,
                        "max_tokens": cot_classifier.max_tokens,
                        "tokens_used": metrics.get("tokens_used"),
                        "prompt_tokens": metrics.get("prompt_tokens"),
                        "completion_tokens": metrics.get("completion_tokens"),
                        "latency": metrics.get("latency"),
                        "strategy": strategy,
                    }
                    rows.append(results)
    
                    raw_resp = metrics.get("raw_response", "")
                    steps, analysis, final_line = cot_classifier._parse_steps_and_final(raw_resp)
                    reasoning_detail = {
                        "sample_id": idx,
                        "raw_response": raw_resp,
                        "parsed_steps": steps,
                        "analysis": analysis,       
                        "final_line": final_line,
                        "final_label": predicted_label,
                        "mapped_label": mapped_label,
                    }
                    detailed_reasoning.append(reasoning_detail)
    
                except Exception as e:
                    print(f"Error processing sample {idx}: {e}")
                    continue
    
            if not rows:
                print(f"No successful classifications for {case_name}")
                continue
    
            output_file = f"results/{model_filename}/cot/role_playing_ethnics/{profile_n}_{role_playing_mode}/results_{case_name.lower()}_{strategy}_cot.csv"
            reasoning_file = f"results/{model_filename}/cot/role_playing_ethnics/{profile_n}_{role_playing_mode}/reasoning_{case_name.lower()}_{strategy}_cot.json"

            os.makedirs(os.path.dirname(output_file), exist_ok=True)
            os.makedirs(os.path.dirname(reasoning_file), exist_ok=True)
    
            df_out = pd.DataFrame(rows)
            df_out.to_csv(output_file, index=False)
            print(f"=== Saved {len(df_out)} rows to {output_file} ===")
    
            with open(reasoning_file, 'w', encoding='utf-8') as f:
                json.dump(detailed_reasoning, f, indent=2, ensure_ascii=False)
            print(f"=== Saved detailed reasoning to {reasoning_file} ===")
    

            try:
                if case_name.lower() == "manipulation":
                    y_true = df_out["true_label"].astype(int)
                    y_pred = df_out["pred_label"].astype(int)
                elif case_name.lower() == "stereotype":
                    y_true = df_out["true_label"].astype(str).str.strip().str.lower()
                    y_pred = df_out["pred_label"].astype(str).str.strip().str.lower()
    
                print(f"\n=== Classification Report for {case_name} ({strategy}) ===")
                print(classification_report(y_true, y_pred, zero_division=0))
                print(f"\n=== Confusion Matrix for {case_name} ===")
                labels = sorted(set(y_true) | set(y_pred))
                print(pd.DataFrame(confusion_matrix(y_true, y_pred, labels=labels), index=labels, columns=labels))
    
                accuracy = (y_true == y_pred).mean()
                print(f"\n=== Accuracy for {case_name}: {accuracy:.2%} ===")
    
                print(f"\n=== Label Distribution ===")
                print("True labels:")
                print(pd.Series(y_true).value_counts())
                print("Predicted labels:")
                print(pd.Series(y_pred).value_counts())
    
            except Exception as e:
                print(f"Error in evaluation for {case_name}: {e}")
    
            metrics = cot_classifier.get_metrics()
            print(f"\n=== CoT Performance Metrics ===")
            print(f"- Avg tokens per call: {metrics['avg_tokens_per_call']:.1f}")
            print(f"- Avg latency per call: {metrics['avg_latency_per_call']:.2f}s")
            print(f"- Total calls: {metrics['total_calls']}")
    
    
            print(f"\n=== Sample Reasoning (raw/parsed) ===")
            for i, reasoning in enumerate(detailed_reasoning[:3]):
                print(f"\nSample {reasoning['sample_id']}:")
                print(f"True Label: {data.loc[reasoning['sample_id']][case.label_col]}")
                print(f"Predicted: {reasoning['final_label']} -> Mapped: {reasoning['mapped_label']}")
                if reasoning['parsed_steps']:
                    print("Parsed Steps:")
                    for step in reasoning['parsed_steps']:
                        print(f"  Step {step['step']}: {step['content'][:150]}...")
                else:
                    print("No explicit 'Step N' blocks found.")
                if reasoning['final_line']:
                    print(f"Final Line: {reasoning['final_line']}")
                print("Raw Response (truncated):")
                print((reasoning['raw_response'] or "")[:300] + "...")
                print("-" * 50)

    
except Exception as e:
    print(f"== Error: {e} ==")
    import traceback
    traceback.print_exc()

print("\n=== Chain of Thought Evaluation Complete ===")


=== Running Chain of Thought for stereotype (optimized) - profile4 ===


Processing stereotype: 100%|██████████| 500/500 [21:22<00:00,  2.56s/it]


=== Saved 500 rows to results/openai_4.1_mini/cot/role_playing_ethnics/profile4_passive/results_stereotype_optimized_cot.csv ===
=== Saved detailed reasoning to results/openai_4.1_mini/cot/role_playing_ethnics/profile4_passive/reasoning_stereotype_optimized_cot.json ===

=== Classification Report for stereotype (optimized) ===
              precision    recall  f1-score   support

  stereotype       0.72      0.80      0.76       250
   unrelated       0.77      0.69      0.73       250

    accuracy                           0.74       500
   macro avg       0.75      0.74      0.74       500
weighted avg       0.75      0.74      0.74       500


=== Confusion Matrix for stereotype ===
            stereotype  unrelated
stereotype         199         51
unrelated           77        173

=== Accuracy for stereotype: 74.40% ===

=== Label Distribution ===
True labels:
true_label
unrelated     250
stereotype    250
Name: count, dtype: int64
Predicted labels:
pred_label
stereotype    276

Processing stereotype: 100%|██████████| 500/500 [20:20<00:00,  2.44s/it]

=== Saved 500 rows to results/openai_4.1_mini/cot/role_playing_ethnics/profile5_passive/results_stereotype_optimized_cot.csv ===
=== Saved detailed reasoning to results/openai_4.1_mini/cot/role_playing_ethnics/profile5_passive/reasoning_stereotype_optimized_cot.json ===

=== Classification Report for stereotype (optimized) ===
              precision    recall  f1-score   support

  stereotype       0.71      0.78      0.74       250
   unrelated       0.75      0.68      0.71       250

    accuracy                           0.73       500
   macro avg       0.73      0.73      0.73       500
weighted avg       0.73      0.73      0.73       500


=== Confusion Matrix for stereotype ===
            stereotype  unrelated
stereotype         194         56
unrelated           81        169

=== Accuracy for stereotype: 72.60% ===

=== Label Distribution ===
True labels:
true_label
unrelated     250
stereotype    250
Name: count, dtype: int64
Predicted labels:
pred_label
stereotype    275